# Inundation model to plot shorelines under each sea level rise scenario. The sea level rise model predicts the shoreline positions under sea level rise ranging from 0 to 10 feet at an interval of 0.1 feet

In [1]:
from osgeo import gdal

In [11]:
import json

In [3]:
import numpy as np
import rasterio

In [ ]:
import rasterio
from rasterio.enums import Resampling

def downscale_raster(input_path, output_path, downscale_factor):
    """
    Downscale a raster by a given factor.

    :param input_path: Path to the input raster file.
    :param output_path: Path to the output downscaled raster file.
    :param downscale_factor: The factor by which to downscale (e.g., 2, 3, etc.).
    """
    with rasterio.open(input_path) as src:
        # Calculate the new dimensions
        new_width = int(src.width / downscale_factor)
        new_height = int(src.height / downscale_factor)

        # Update the metadata (transform and dimensions)
        transform = src.transform * src.transform.scale(
            (src.width / new_width),
            (src.height / new_height)
        )
        metadata = src.meta.copy()
        metadata.update({
            "driver": "GTiff",
            "height": new_height,
            "width": new_width,
            "transform": transform
        })

        # Read and resample the data
        data = src.read(
            out_shape=(
                src.count,
                new_height,
                new_width
            ),
            resampling=Resampling.bilinear
        )

        # Write the resampled data to the output file
        with rasterio.open(output_path, 'w', **metadata) as dst:
            dst.write(data)

# Example usage
input_raster = '/Users/ziyangliu/Downloads/WA_SEW_dems/WA_GCS_5m_NAVD88m_merged.tif'
output_raster = '/Users/ziyangliu/Downloads/WA_SEW_dems/WA_GCS_5m_NAVD88m_merged_downscale.tif'
downscale_factor = 4  # Example: downscale by a factor of 4

downscale_raster(input_raster, output_raster, downscale_factor)


In [6]:
def load_dem(file_path):
    """Load a DEM file and return the elevation data."""
    with rasterio.open(file_path) as dem:
        return dem.read(1)

In [7]:
dem_file_path = '/Users/ziyangliu/Downloads/WA_SEW_dems/WA_GCS_5m_NAVD88m_merged_downscale.tif'

#Load DEM data
elevation = load_dem(dem_file_path)

elevation

array([[-4.1075468e-01, -4.1075468e-01, -4.1075468e-01, ...,
         7.3620496e+00,  9.7138720e+00,  1.0996420e+01],
       [-5.0000000e-01, -5.0000000e-01, -5.0000000e-01, ...,
         8.3248253e+00,  9.0251217e+00,  9.3487225e+00],
       [-5.0000000e-01, -5.0000000e-01, -5.0000000e-01, ...,
         8.3596449e+00,  8.6027555e+00,  8.3225584e+00],
       ...,
       [-9.9990000e+03, -9.9990000e+03, -9.9990000e+03, ...,
        -5.8469504e-01, -5.6501812e-01, -4.2404103e-01],
       [-9.9990000e+03, -9.9990000e+03, -9.9990000e+03, ...,
        -5.4489279e-01, -6.0977536e-01, -5.2297431e-01],
       [-5.7158687e+03, -5.7158687e+03, -5.7158687e+03, ...,
        -2.7271467e-01, -2.7555797e-01, -2.5335377e-01]], dtype=float32)

In [8]:
# get the metadata of the DEM file
with rasterio.open(dem_file_path) as dem:
    metadata = dem.meta

metadata

ERROR 1: PROJ: proj_identify: /usr/local/Caskroom/miniconda/base/envs/geo/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.


{'driver': 'GTiff',
 'dtype': 'float32',
 'nodata': None,
 'width': 1420,
 'height': 1325,
 'count': 1,
 'crs': CRS.from_wkt('GEOGCS["NAD83",DATUM["North American Datum 1983",SPHEROID["GRS 1980",6378137,298.257222101004]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]'),
 'transform': Affine(0.00017999999999999969, 0.0, -124.16647693837503,
        0.0, -0.0001800679245283019, 46.92608299379318)}

In [ ]:
elevation.shape

In [ ]:
# #update the metadata
# metadata['height'] = elevation.shape[0]
# metadata['width'] = elevation.shape[1]

# #update the transform
# metadata['transform'] = rasterio.Affine(metadata['transform'][0],metadata['transform'][1],metadata['transform'][2],metadata['transform'][3],metadata['transform'][4],metadata['transform'][5]+metadata['transform'][4]*1)

# metadata

In [ ]:
def find_coastal_cells(elevation_data):
    """Find coastal cells in the DEM data."""
    coastal_cells = []
    rows, cols = elevation_data.shape
    for row in range(rows):
        for col in range(cols):
            # Check if the current cell is at sea level
            if elevation_data[row, col] <= 0:
                # If it is at sea level,
                # then check neighboring cells to determine if it's a coastal cell
                for d_row, d_col in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    r, c = row + d_row, col + d_col

                    # to make sure it is not out of bound
                    if 0 <= r < rows and 0 <= c < cols:

                        # if the neighboring cell is above sea level
                        # then the current cell is a coastal cell
                        if elevation_data[r, c] > 0:
                            coastal_cells.append((row, col))
                            break
    return coastal_cells

In [ ]:
#sea_level = -0.5  # Sea level is set at -0.5 m in our DEM

coastal_cells = find_coastal_cells(elevation)
coastal_cells

In [ ]:
len(coastal_cells)

In [ ]:
elevation

In [ ]:
def flood_fill(start_points, elevation_data, sea_level_rise):
    """Flood fill algorithm to determine areas connected to the sea.
    Sea will not be determined as inundated, only cells that were not sea but
    are wet after the sea level rise are considered inundated"""

    # Create a boolean array to keep track of which cells are inundated
    # and initialize it to False
    inundated = np.zeros_like(elevation_data, dtype=bool)

    # Loop through the start points
    # each start point is a coastal cell
    for point in start_points:

        # Create a stack to store cells that are potentially inundated and thus needs to be checked

        stack = [point]
        #print(point)


        # this while loop will:
        # 1. pop a cell from the stack
        # 2. check if it is inundated
        # 3. if it is, then append its neighbors to the stack
        # 4. repeat 1-3 until the stack is empty
        while stack:
            #print(stack)

            # Pop a cell from the stack
            x, y = stack.pop()
            # if it is not inundated yet and its elevation is below sea level rise
            if not inundated[x, y] and elevation_data[x, y] <= sea_level_rise and elevation_data[x, y] > -1000:

                # then it is inundated, set to true
                inundated[x, y] = True
                # and check its four neighbors
                for nx, ny in [(x-1, y), (x+1, y), (x, y-1), (x, y+1)]:
                    if 0 <= nx < elevation_data.shape[0] and 0 <= ny < elevation_data.shape[1]:
                        stack.append((nx, ny))

    #only inundated cells that were not sea before are considered inundated
    inundated=np.where(elevation_data<0,False,inundated)
    return inundated


In [ ]:
elevation[1,938]

In [ ]:
flooded_cell_one_foot=flood_fill(coastal_cells, elevation, 0.3048)
flooded_cell_one_foot

In [ ]:
np.any(flooded_cell_one_foot==True)

In [ ]:
#count the number of flooded cells
np.sum(flooded_cell_one_foot==True)

In [ ]:
np.sum(flooded_cell_one_foot==False)

In [ ]:
a=np.array([[1,2,3],[4,5,6],[7,8,9]])
b=np.array([[True,False,True],[False,True,False],[False,True,True]])
# set the cells that are True in b to -0.5 in a

mask = np.where(b, -0.5, a)

mask = np.where(~b, a-1, mask)

mask


In [ ]:
# update the dem, and set the elevation of flooded cells to -0.5
def update_dem(elevation_data, flooded_cells, sea_level_rise):
    """Update the DEM data with flooded cells."""


    # if a cell is flooded, set its elevation to -0.5
    mask = np.where(flooded_cells, -0.5, elevation_data)



    # if a cell is not -9999 and not flooded (valid cell not flooded, including sea cell), subtract the sea level rise from its elevation
    mask = np.where(~flooded_cells & (elevation_data > -1000), elevation_data - sea_level_rise, mask)

    # if a cell is not flooded, but its elevation is below zero (sea cell), set its elevation to -0.5
    mask = np.where(~flooded_cells & (elevation_data <= 0), -0.5, mask)

    #mask = np.where(~flooded_cells, elevation_data - sea_level_rise, mask)



    return mask


In [ ]:
update_dem(elevation, flooded_cell_one_foot, 0.3048)

In [ ]:
elevation

In [ ]:
np.nanmax(elevation[-1])

In [9]:
# find the coordinates of a cell
def find_coordinates(cell):
    """Find the coordinates of a cell in the DEM data."""
    origin_lon=metadata["transform"][2]
    origin_lat=metadata["transform"][5]
    lon = origin_lon + cell[1] * metadata["transform"][0]
    lat = origin_lat + cell[0] * metadata["transform"][4]
    return (lon, lat)

#Use example
find_coordinates((2000/4, 3000/4))  #(row, column)

(-124.03147693837502, 46.83604903152903)

In [ ]:
import os
# test if the path exists
os.path.exists('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/dem')

In [ ]:
import matplotlib.pyplot as plt
from osgeo import gdal

In [ ]:
import numpy as np

In [ ]:
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/dem/slr_2.0ft.tif')
array = src.read(1)

array = np.where(array == -10002.054, np.nan, array)


plt.figure(figsize=(8, 6))
plt.imshow(array, cmap='terrain',vmin=np.nanmin(array),vmax=np.nanmax(array))  # Use 'terrain' colormap for elevation
plt.colorbar(label='Elevation (meters)')
plt.title('Digital Elevation Model of 2 feet slr')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [ ]:
elevation = np.where(elevation == -9999, np.nan, elevation)
plt.figure(figsize=(8, 6))
plt.imshow(elevation, cmap='terrain',vmin=np.nanmin(elevation),vmax=np.nanmax(elevation))  # Use 'terrain' colormap for elevation
plt.colorbar(label='Boolean values')
plt.title('DEM Orignial Data')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [ ]:
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_10.0ft.tif')
array = src.read(1)




plt.figure(figsize=(8, 6))
plt.imshow(array, cmap='gray_r')  # Use 'terrain' colormap for elevation
plt.colorbar(label='Boolean values')
plt.title('Inundated Area of 10.0 feet slr')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [ ]:
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_0.1ft.tif')
array = src.read(1)




plt.figure(figsize=(8, 6))
plt.imshow(flooded_cell_one_foot, cmap='gray_r')  # Use 'terrain' colormap for elevation
plt.colorbar(label='Boolean values')
plt.title('Inundated Area of 0.1 feet slr')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [ ]:
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/uncertainty/slr_0.1_10.0ft.tif')
#src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_10.0ft.tif')
array_1 = src.read(1)




plt.figure(figsize=(8, 6))
plt.imshow(array_1, cmap='gray_r')  # Use 'terrain' colormap for elevation
plt.colorbar(label='Boolean values')
plt.title('Inundated Area of 10.0 feet slr')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [ ]:
np.nanmin(array_1)

In [ ]:
# test if any of the array_1 cell is 1
np.any(array_1 == 1)


In [ ]:
# write a function: Suppose I have two inundated arrays, now create a new array, if cell in two inundated arrays are both True, then set it to 2, if the cell in the first inundated array is True but in the second inundated array is False, then set it to 1, if the cell in both inundated arrays are False, then set it to 0
# inundated2 is the more severe one
def compare_inundated(inundated1, inundated2):
    """Compare two inundated arrays."""
    # create a new array
    new_array = np.zeros_like(inundated1)
    # if the cell in both inundated arrays are True, then set it to 2
    new_array = np.where(((inundated1==1) & (inundated2==1)), 2, new_array)
    # if one of them is 1 and the other is 0, then set it to 1
    new_array = np.where(((inundated1==0) & (inundated2==1)), 1, new_array)


    # if the cell in both inundated arrays are False, then set it to 0
    #new_array = np.where(((inundated1==0) & (inundated2==0)), 0, new_array)
    return new_array


In [ ]:
# let's test the compare_inundated function with slr of 0.1ft and 10.0ft

# first, load the inundated arrays of 0.1ft and 10.0ft
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_0.1ft.tif')
array_1 = src.read(1)
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_10.0ft.tif')
array_2 = src.read(1)

# compare the two inundated arrays
new_array = compare_inundated(array_1, array_2)

# plot the new array
plt.figure(figsize=(8, 6))
plt.imshow(new_array, cmap='gray_r')  # Use 'terrain' colormap for elevation
plt.colorbar(label='Boolean values')
plt.title('Inundated Area of 10.0 feet slr')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()


In [ ]:
# first we need to create those two inundated arrays, then we open them using rasterio

# create all inundated arrays
for slr in range(1, 101):
    # divide the slr by 10 to get the actual feet
    slr = slr/10
    # convert the sea level rise from feet to meters
    slr_m = slr * 0.3048

    # The DEM refers to NAVD88, I need to refert it to MHHW
    # the difference between NAVD88 and MHHW is 8.02 feet
    slr_m = slr_m + 8.02*0.3048
    # use the flood_fill function to find inundated cells
    inundated = flood_fill(coastal_cells, elevation, slr_m)

    # use the update_dem function to update the DEM data

    new_dem = update_dem(elevation, inundated, slr_m)

    # find the coastal cells at the new sea level
    # coastal_cells_new = find_coastal_cells(new_dem, -0.5)

    # # find the coordinates of the new coastal cells
    # coordinates = [find_coordinates(cell) for cell in coastal_cells_new]

    # write the new DEM data to a new file
    with rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/dem/slr_{slr}ft.tif', 'w', **metadata) as dst:
        dst.write(new_dem, 1)

    # write the inundated cells to a new file
    with rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_{slr}ft.tif', 'w', **metadata) as dst:
        dst.write(inundated.astype(np.uint8), 1)

    print(slr)



In [ ]:
import json

for i in range(1,100): # 100
    ii=i/10

    #open the first inundated array
    src1 = rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_{ii}ft.tif')
    array1 = src1.read(1)

    for j in range(i+1,101):   # 101
        jj=j/10
        #open the second inundated array
        src2 = rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_{jj}ft.tif')
        array2 = src2.read(1)

        #compare the two inundated arrays
        new_array=compare_inundated(array1, array2)

        with rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/uncertainty/slr_{ii}_{jj}ft.tif', 'w', **metadata) as dst:
            dst.write(new_array.astype(np.uint8), 1)

         # Find the indices where the value is 2
        indices_2 = np.where(new_array == 2)

        # Find the indices where the value is 1
        indices_1 = np.where(new_array == 1)

        list_of_indices_2 = list(zip(indices_2[0], indices_2[1]))
        list_of_indices_1 = list(zip(indices_1[0], indices_1[1]))

        coords_2=[find_coordinates(cell) for cell in list_of_indices_2]
        coords_1=[find_coordinates(cell) for cell in list_of_indices_1]

        rounded_coords_2 = [[round(num, 5) for num in inner_list] for inner_list in coords_2]
        rounded_coords_1 = [[round(num, 5) for num in inner_list] for inner_list in coords_1]
        feature2={
            "type": "Feature",
            "geometry": {
                "type": "MultiPoint",
                "coordinates": rounded_coords_2
            },
            "properties": {
                "layer": "layer_2"
            }
        }

        feature1 = {
            "type": "Feature",
            "geometry": {
                "type": "MultiPoint",
                "coordinates": rounded_coords_1
            },
            "properties": {
                "layer": "layer_1"
            }
        }

        # Combine the features into a FeatureCollection
        geojson_feature_collection = {
            "type": "FeatureCollection",
            "features": [feature1, feature2]
        }

        # Save the FeatureCollection to a file
        with open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/vector/uncertainty/slr_{ii}_{jj}ft.geojson', 'w') as file:
            json.dump(geojson_feature_collection, file, indent=4)

In [12]:
# test cell to generate slr_0.1_1.5ft.geojson
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/uncertainty/slr_0.1_1.5ft.tif')
array = src.read(1)
indices_1 = np.where(array == 1)
indices_2 = np.where(array == 2)
list_of_indices_2 = list(zip(indices_2[0], indices_2[1]))
list_of_indices_1 = list(zip(indices_1[0], indices_1[1]))

coords_2=[find_coordinates(cell) for cell in list_of_indices_2]
coords_1=[find_coordinates(cell) for cell in list_of_indices_1]

rounded_coords_2 = [[round(num, 5) for num in inner_list] for inner_list in coords_2]
rounded_coords_1 = [[round(num, 5) for num in inner_list] for inner_list in coords_1]
feature2={
    "type": "Feature",
    "geometry": {
        "type": "MultiPoint",
        "coordinates": rounded_coords_2
    },
    "properties": {
        "layer": "layer_2"
    }
}

feature1 = {
    "type": "Feature",
    "geometry": {
        "type": "MultiPoint",
        "coordinates": rounded_coords_1
    },
    "properties": {
        "layer": "layer_1"
    }
}

# Combine the features into a FeatureCollection
geojson_feature_collection = {
    "type": "FeatureCollection",
    "features": [feature1, feature2]
}

# Save the FeatureCollection to a file
with open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/vector/uncertainty/slr_0.1_1.5ft.geojson', 'w') as file:
    json.dump(geojson_feature_collection, file, indent=4)


In [13]:
def geojsonUncertainty(low,high):
    openUrl='/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/uncertainty/slr_'+str(low)+'_'+str(high)+'ft.tif'
    src = rasterio.open(openUrl)
    array = src.read(1)
    indices_1 = np.where(array == 1)
    indices_2 = np.where(array == 2)
    list_of_indices_2 = list(zip(indices_2[0], indices_2[1]))
    list_of_indices_1 = list(zip(indices_1[0], indices_1[1]))

    coords_2=[find_coordinates(cell) for cell in list_of_indices_2]
    coords_1=[find_coordinates(cell) for cell in list_of_indices_1]

    rounded_coords_2 = [[round(num, 5) for num in inner_list] for inner_list in coords_2]
    rounded_coords_1 = [[round(num, 5) for num in inner_list] for inner_list in coords_1]
    feature2={
        "type": "Feature",
        "geometry": {
            "type": "MultiPoint",
            "coordinates": rounded_coords_2
        },
        "properties": {
            "layer": "layer_2"
        }
    }

    feature1 = {
        "type": "Feature",
        "geometry": {
            "type": "MultiPoint",
            "coordinates": rounded_coords_1
        },
        "properties": {
            "layer": "layer_1"
        }
    }

    # Combine the features into a FeatureCollection
    geojson_feature_collection = {
        "type": "FeatureCollection",
        "features": [feature1, feature2]
    }

    saveUrl='/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/vector/uncertainty/slr_'+str(low)+'_'+str(high)+'ft.geojson'
    # Save the FeatureCollection to a file
    with open(saveUrl, 'w') as file:
        json.dump(geojson_feature_collection, file, indent=4)


In [14]:
geojsonUncertainty(0.1,1.3)

In [15]:
listOfUncertaintyBounds=[
    [
        -0.1,
        0.2
    ],
    [
        0,
        0.3
    ],
    [
        0,
        0.4
    ],
    [
        0,
        0.6
    ],
    [
        0,
        0.5
    ],
    [
        0,
        0.8
    ],
    [
        0.1,
        1.1
    ],
    [
        0,
        1
    ],
    [
        0,
        1.1
    ],
    [
        0.2,
        1.5
    ],
    [
        0.1,
        1.3
    ],
    [
        0.1,
        1.5
    ],
    [
        0.3,
        1.9
    ],
    [
        0.1,
        1.6
    ],
    [
        0.1,
        1.9
    ],
    [
        0.4,
        2.4
    ],
    [
        0.1,
        2
    ],
    [
        0.1,
        2.4
    ],
    [
        0.5,
        3
    ],
    [
        0.1,
        3
    ],
    [
        0.5,
        3.2
    ],
    [
        0.1,
        2.7
    ],
    [
        0.1,
        3.2
    ],
    [
        0.7,
        3.9
    ],
    [
        0.1,
        3.1
    ],
    [
        0.1,
        3.9
    ],
    [
        0.7,
        4.5
    ],
    [
        0.1,
        3.6
    ],
    [
        0.1,
        4.5
    ],
    [
        0.8,
        5.1
    ],
    [
        0,
        4.1
    ],
    [
        0,
        5.1
    ],
    [
        0.8,
        5.8
    ],
    [
        -0.1,
        4.6
    ],
    [
        -0.1,
        5.8
    ]
]

In [18]:
for bounds in listOfUncertaintyBounds:
    if bounds[0]<=0:
        continue
    else:
        geojsonUncertainty(float(bounds[0]), float(bounds[1]))

In [ ]:
# open the inundated arrays of 0.1 feet and 0.2 feet
src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_0.1ft.tif')
array_1 = src.read(1)
array_1

src = rasterio.open('/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_0.2ft.tif')
array_2 = src.read(1)
array_2

In [ ]:
comp_array=compare_inundated(array_1, array_2)

In [ ]:
# plot the comp_array

plt.figure(figsize=(8, 6))
plt.imshow(comp_array, cmap='gray_r')  # Use 'terrain' colormap for elevation
plt.colorbar(label='Boolean values')
plt.title('Uncertain Inundated Area of 0.1 feet and 0.2 feet slr')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [ ]:
# if any of the comp_array cell is 1
np.any(comp_array == 1)
# how many of them are 1
np.sum(comp_array == 1)

In [ ]:
np.sum(comp_array == 2)

In [ ]:
# create a multipoint geojson file
from shapely.geometry import MultiPoint

import json


In [ ]:
# after creating all inundated arrays, we can open them using rasterio

# nested for loop
for i in range(1,2): # 100
    ii=i/10

    #open the first inundated array
    src1 = rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_{ii}ft.tif')
    array1 = src.read(1)

    for j in range(i+1,3):   # 101
        jj=j/10
        #open the second inundated array
        src2 = rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/inundated/slr_{jj}ft.tif')
        array2 = src.read(1)

        #compare the two inundated arrays
        new_array=compare_inundated(array1, array2)

        # Find the indices where the value is 2
        indices_2 = np.where(new_array == 2)

        # Find the indices where the value is 1
        indices_1 = np.where(new_array == 1)

        list_of_indices_2 = list(zip(indices_2[0], indices_2[1]))
        list_of_indices_1 = list(zip(indices_1[0], indices_1[1]))

        coords_2=[find_coordinates(cell) for cell in list_of_indices_2]
        coords_1=[find_coordinates(cell) for cell in list_of_indices_1]

        rounded_coords_2 = [[round(num, 5) for num in inner_list] for inner_list in coords_2]
        rounded_coords_1 = [[round(num, 5) for num in inner_list] for inner_list in coords_1]
        feature2={
            "type": "Feature",
            "geometry": {
                "type": "MultiPoint",
                "coordinates": rounded_coords_2
            },
            "properties": {
                "layer": "layer_2"
            }
        }

        feature1 = {
            "type": "Feature",
            "geometry": {
                "type": "MultiPoint",
                "coordinates": rounded_coords_1
            },
            "properties": {
                "layer": "layer_1"
            }
        }

        # Combine the features into a FeatureCollection
        geojson_feature_collection = {
            "type": "FeatureCollection",
            "features": [feature1, feature2]
        }

        # Save the FeatureCollection to a file
        with open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/vector/uncertainty/slr_{ii}_{jj}ft.geojson', 'w') as file:
            json.dump(geojson_feature_collection, file, indent=4)







        #write the new array to a new file
        # with rasterio.open(f'/Users/ziyangliu/Desktop/Workspace/Grays_Harbor_Storymap/assets/slr_results/raster/uncertainty/slr_{ii}_{jj}ft.tif', 'w', **metadata) as dst:
        #     dst.write(new_array, 1)

In [ ]:
import numpy as np

# Example array
array = np.array([[0, 1, 2],
                  [2, 0, 1],
                  [1, 2, 0]])

# Find the indices where the value is 2
indices = np.where(array == 2)

# Convert indices to a list of tuples
list_of_indices = list(zip(indices[0], indices[1]))

print(list_of_indices)  # This will print the list of indices where the value is 2


In [ ]:
[find_coordinates(cell) for cell in list_of_indices]

In [ ]:
indices

In [ ]:
c=0
for i in range(1, 100):
    # due to floating-point precision, I need to round the sea level rise to
    #  1 decimal places
    ii=i/10

    for j in range(i+1,101,1):
        jj=j/10
        c=c+1

        print(ii,jj)
print(c)


In [ ]:
# test
round(4.3999999999999995, 1)

In [ ]:
# test
for i in np.arange(0.1, 10.1, 0.1):
    print(round(i,1))

In [ ]:

# Usage Example

sea_level = 0  # Assuming sea level is at 0m in the DEM
sea_level_rise = 1  # 1 meter sea level rise scenario


# Find coastal cells
coastal_points = find_coastal_cells(elevation, sea_level)

# Apply flood fill to find inundated areas
inundated_areas = flood_fill(coastal_points, elevation, sea_level_rise)
